# Phase 2 · Notebook 6 — Train & honestly validate XGBoost

We train a gradient-boosted tree model to predict flood from terrain, and — most importantly — we validate it the *honest* way.

**The trap:** geospatial data is spatially auto-correlated (neighbouring pixels are nearly identical). Ordinary random cross-validation puts a pixel in training and its neighbour in testing, so the model looks far better than it really is. We show this by running **both** random and **spatial-block** cross-validation and comparing.

## Step 1 — Load the training table

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name in ("phase2", "notebooks") else Path.cwd()
P2 = REPO / "phase2"
df = pd.read_csv(P2 / "training_data.csv")

FEATURES = ["elevation", "slope", "dist_river", "builtup", "hand", "curvature", "local_relief"]
X = df[FEATURES].values
y = df["label"].values
groups = df["block"].values
print("Training rows:", len(df), "| features:", len(FEATURES))

def new_model():
    return xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.08,
                             subsample=0.9, colsample_bytree=0.9, n_jobs=4,
                             eval_metric="logloss", random_state=0)

Training rows: 80000 | features: 7


## Step 2 — Random vs. spatial cross-validation

- **Random CV:** shuffle all pixels, 5 folds. *Optimistic* — leaks neighbours across folds.
- **Spatial CV:** split by the 5×5 grid blocks (`GroupKFold`), so an entire region is held out at a time. *Honest* — the model is tested on ground it has never seen nearby.

The spatial number is the one to trust and report.

In [2]:
rand_auc = []
for tr, te in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
    m = new_model().fit(X[tr], y[tr])
    rand_auc.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))

spat_auc = []
for tr, te in GroupKFold(5).split(X, y, groups=groups):
    m = new_model().fit(X[tr], y[tr])
    spat_auc.append(roc_auc_score(y[te], m.predict_proba(X[te])[:, 1]))

print(f"Random  CV AUC: {np.mean(rand_auc):.3f} ± {np.std(rand_auc):.3f}   (optimistic)")
print(f"Spatial CV AUC: {np.mean(spat_auc):.3f} ± {np.std(spat_auc):.3f}   (HONEST — report this)")

Random  CV AUC: 0.971 ± 0.001   (optimistic)
Spatial CV AUC: 0.925 ± 0.016   (HONEST — report this)


## Step 3 — Confusion matrix on a held-out spatial fold

We hold out one block-fold, predict it, and threshold at 0.5 to see real-world hits and misses.

In [3]:
tr, te = next(GroupKFold(5).split(X, y, groups=groups))
m = new_model().fit(X[tr], y[tr])
pred = (m.predict_proba(X[te])[:, 1] >= 0.5).astype(int)
cm = confusion_matrix(y[te], pred)
print("Held-out spatial fold:")
print(f"   AUC: {roc_auc_score(y[te], m.predict_proba(X[te])[:,1]):.3f} | F1: {f1_score(y[te], pred):.3f}")
print("   confusion matrix [[TN FP] [FN TP]]:\n", cm)

Held-out spatial fold:
   AUC: 0.903 | F1: 0.976
   confusion matrix [[TN FP] [FN TP]]:
 [[  579   425]
 [  276 14263]]


## Step 4 — Train the final model and explain it with SHAP

We fit on all the data and save the model. **SHAP** then tells us *which features drive the predictions*. (Note the honest caveat: `builtup` ranks high partly because the SAR labels can't see flooding *inside* built-up areas, so the model partly learns “built-up → dry”.)

In [4]:
import shap

model = new_model().fit(X, y)
model.save_model(P2 / "flood_model.json")

sample = X[np.random.default_rng(0).choice(len(X), 4000, replace=False)]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)

plt.figure()
shap.summary_plot(shap_values, sample, feature_names=FEATURES, show=False)
plt.tight_layout()
plt.savefig(REPO / "data" / "outputs" / "shap_summary.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved model -> flood_model.json and SHAP plot -> shap_summary.png")

Saved model -> flood_model.json and SHAP plot -> shap_summary.png


/var/folders/jr/0tppzf_n6rscbz7qzblfgwbc0000gn/T/ipykernel_16411/1794456287.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Recap

We measured the model both ways (random vs spatial CV), saw the spatial-CV drop that proves why it matters, checked a confusion matrix, trained the final model, and explained it with SHAP.

**Next — Notebook 7:** apply this model to every pixel to make a full ML susceptibility map.